In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

import numpy as np
import sionna
# For link-level simulations
from sionna.phy.channel import OFDMChannel, CIRDataset
from sionna.phy.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver
from sionna.phy.utils import ebnodb2no, PlotBER
from sionna.phy.ofdm import KBestDetector, LinearDetector

from sionna.phy.mimo import StreamManagement
from mitsuba import Point3f
# Import Sionna RT components
from sionna.rt import load_scene, Camera, Transmitter, Receiver, PlanarArray,\
                      PathSolver, RadioMapSolver

###############################################################################

In [ ]:
# Global parameters
###############################################################################
F_C            = 3.5e9       # carrier [Hz]
CELL_SIZE      = (5., 5.)    # radio‑map grid [m]
SAMPLES_PER_TX = 2_000_000   # rays per Tx
MAX_DEPTH      = 5           # max reflections
FFT_SIZE       = 256
NUM_SYM        = 14
SUBC_SP        = 30e3        # subcarrier spacing [Hz]
BATCH_BER      = 512         # MC batch for BER
EBNO_DB        = 5.0         # Eb/N0 for BER curve
RX_REF_POS     = np.array([50., 50., 1.5])  # reference Rx

###############################################################################
# Scene helpers
###############################################################################

def make_scene(jam2_pos=None):
    """Return a Munich scene with Tx0 + Jam1 (+ optional Jam2)"""
    scene = load_scene(sionna.rt.scene.munich)
    scene.frequency = F_C

    # Common ULAs
    scene.tx_array = PlanarArray(num_rows=1,num_cols=1,pattern="iso", polarization="V")
    scene.rx_array = scene.tx_array

    # Serving base‑station
    scene.remove("tx0")  # remove default Tx0
    scene.add(Transmitter("tx0", position=[0., 0., 50.], power_dbm=43))

    # Static jammer
    scene.remove("jam1")  # remove default Jam1
    scene.add(Transmitter("jam1", position=[150., 80., 50.], power_dbm=30))
    
 
    # Mobile jammer (optional)
    if jam2_pos is not None:
        # unpack to built‑in floats
        x, y, z = map(float, jam2_pos)
        scene.remove("jam2")
        scene.add(
            Transmitter(
                "jam2",
                position=Point3f(x, y, z),
                power_dbm=30,
                velocity=Point3f(0.0, 0.0, 0.0)
            )
        )
    
    return scene

###############################################################################
# Radio‑map utilities
###############################################################################

def compute_radiomap(scene):
    rms = RadioMapSolver()
    rm  = rms(scene, cell_size=CELL_SIZE, max_depth=MAX_DEPTH,
              samples_per_tx=SAMPLES_PER_TX)
    return rm

###############################################################################
# BER for a single Rx position (RayTracingChannel, dynamic)
###############################################################################



###############################################################################
# Heat‑map generation
###############################################################################

def make_heatmaps():
    print(">>> Computing radio‑maps (no‑jam vs with‑jam)…")
    # --- (A) no‑jam ---------------------------------------------------------
    scene_no = make_scene(jam2_pos=None)
    rm_no    = compute_radiomap(scene_no)

    # --- (B) with jam (static+mobile at initial pos) ------------------------
    scene_j  = make_scene(jam2_pos=[-200., -100., 50.])
    rm_j     = compute_radiomap(scene_j)

    # RSS tensors ------------------------------------------------------------
    rss_sig_no = rm_no.rss[0]
    rss_sig_j  = rm_j.rss[0]
    rss_int_j  = np.max(rm_j.rss[1:], axis=0)  # max over 2 jammers
    

    # J/S heat‑map -----------------------------------------------------------
    js_map = rss_int_j - rss_sig_j
   
    plt.contour(js_map, levels=[0], colors="k")
    plt.savefig("js_heatmap.png", dpi=200)

    # SINR -------------------------------------------------------------------
    sinr_no = rm_no.sinr[0]
    sinr_j  = rm_j.sinr[0]
    delta   = sinr_j - sinr_no
    
    print(sinr_no.shape)
    print(sinr_no)
    print(sinr_j.shape)
    print(sinr_j)
    print(delta)
    # Plot SINR heat‑map
    plt.figure()
    plt.contourf(delta, levels=np.arange(-20, 31, 1), cmap="magma")
    plt.colorbar(label="SINR (dB)")
    plt.title("SINR with Jam (dB)")
    plt.xlabel("x [m]")
    plt.ylabel("y [m]")
    plt.xlim(0, 200)
    plt.ylim(0, 200)
    


    rm_j.show(metric="sinr",     tx=0, vmin=-20, vmax=30)
    plt.savefig("sinr_nojam.png", dpi=200)
    # 如果想看「Receiver 永遠選 SINR 最佳的基地台」：
    
    rm_no.show(metric="sinr",     tx=0, vmin=-20, vmax=30)

    
    # CDF of the SINR for transmitter 0
    rm_no.cdf(metric="sinr", tx=0)
    rm_j.cdf(metric="sinr", tx=0)
    plt.savefig("delta_sinr.png", dpi=200)


###############################################################################
if __name__ == "__main__":
    tf.random.set_seed(1)
    np.random.seed(1)
    make_heatmaps()

    print("\nAll figures saved to current directory.")